# Unit 5: 高级 CNN 技术

## 学习目标
- 理解 Batch Normalization 的原理和效果
- 掌握 Dropout 正则化
- 实现残差连接 (Residual Connection)
- 对比不同学习率调度策略
- 了解梯度裁剪的作用

## 5.1 Batch Normalization

**问题**：深度网络训练中，各层输入的分布会随参数更新而变化 (**Internal Covariate Shift**)，导致：
- 需要很小的学习率
- 对初始化敏感
- 训练缓慢

**BatchNorm 的解决方案**：对每个 mini-batch 的每个通道做标准化。

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma^2_B + \epsilon}}$$
$$y = \gamma \hat{x} + \beta$$

其中 $\gamma$ 和 $\beta$ 是**可学习的**缩放和偏移参数。  
相当于让网络“学会”是否需要、以及需要多少缩放与偏移——本质上是给归一化加了一个灵活的线性变换。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import copy

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

### BatchNorm 带来的好处
- 允许使用**更大的学习率**
- **减少对初始化的依赖**
- 有一定的**正则化效果**（mini-batch 统计量带来噪声）
- 加速收敛，通常可以**减少 10-20 倍的训练步数**

### 注意
- `model.train()` 时使用当前 batch 的均值和方差
- `model.eval()` 时使用训练期间累积的**全局**均值和方差（running mean/var）
- BatchNorm 对 batch_size 敏感，太小会不稳定

In [ ]:
x = torch.randn(4, 16, 7, 7)
bn = nn.BatchNorm2d(16)

print(f"Input shape: {x.shape}")
print(f"BN weight shape: {bn.weight.shape}")
print(f"BN bias shape: {bn.bias.shape}")


bn.train()
out_train = bn(x)
print(f"\nTrain mode - mean: {out_train.mean():.4f}, std: {out_train.std():.4f}")

bn.eval()
with torch.no_grad():
    out_eval = bn(x)
    print(f"Eval mode  - mean: {out_eval.mean():.4f}, std: {out_eval.std():.4f}")

### 实验：有/无 BatchNorm 对比

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, use_bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU()]
        if use_bn:
            layers.insert(1, nn.BatchNorm2d(out_c)) # 在激活层之前添加BatchNorm
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class TestCNN(nn.Module):
    def __init__(self, use_bn=True):
        super().__init__()
        self.conv1 = ConvBlock(3, 32, use_bn)
        self.conv2 = ConvBlock(32, 64, use_bn)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(64 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(self.conv1(x))
        x = self.pool(self.conv2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model_no_bn = TestCNN(use_bn=False)
model_with_bn = TestCNN(use_bn=True)
print(f"Without BN: {sum(p.numel() for p in model_no_bn.parameters()):,} params")
print(f"With BN:    {sum(p.numel() for p in model_with_bn.parameters()):,} params (only +{sum(p.numel() for p in model_with_bn.parameters()) - sum(p.numel() for p in model_no_bn.parameters())} from BN)")

## 5.2 Dropout 正则化

训练时**随机丢弃**一部分神经元（置零），防止**过拟合**。

- 每个神经元以概率 $p$ 被保留（PyTorch 中的 $p$ 是丢弃概率）
- 推理时所有神经元都参与，但输出按比例缩放
- 相当于隐式地在训练一个**大型集成模型**

In [ ]:
x = torch.ones(1, 10)
dropout = nn.Dropout(p=0.5)

dropout.train()
out_train = dropout(x)
print(f"Train mode: {out_train}")
print(f"  非零元素数: {(out_train != 0).sum().item()}")
print(f"  保留元素值: ~2.0 (因为训练时按 1/(1-p)=2 缩放)")

dropout.eval()
out_eval = dropout(x)
print(f"\nEval mode:  {out_eval}")
print(f"  推理时不做 Dropout，值保持不变")

In [ ]:
p_vals = np.linspace(0, 0.9, 10)
retained = []
for p in p_vals:
    drop = nn.Dropout(p)
    drop.train()
    out = drop(torch.ones(10000))
    retained.append((out != 0).float().mean().item())

plt.plot(p_vals, retained, "o-")
plt.plot([0, 0.9], [1, 0.1], "r--", label="1-p (理论)")
plt.xlabel("Dropout probability (p)")
plt.ylabel("Fraction of neurons retained")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5.3 残差连接 (Residual Connection)

ResNet 的核心思想：学习残差映射 $F(x) = H(x) - x$ 而非直接学习 $H(x)$。

$$\text{Output} = F(x) + x \quad \text{(shortcut/skip connection)}$$

**为什么有效**：
- 梯度可以通过 shortcut 直接回传，缓解**梯度消失**
- 即使 $F(x)$ 学不到东西，至少可以恒等映射（identity mapping）
- 使得训练 100+ 层网络成为可能

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

x = torch.randn(2, 16, 32, 32)
block_same = ResidualBlock(16, 16)
block_down = ResidualBlock(16, 32, stride=2)

print(f"Input:                 {x.shape}")
print(f"Same shape block:      {block_same(x).shape}")
print(f"Downsample block:      {block_down(x).shape}")

### Mini-ResNet 架构

In [ ]:
class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_channels = 32

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        self.layer1 = self._make_layer(32, 2, stride=1)
        self.layer2 = self._make_layer(64, 2, stride=2)
        self.layer3 = self._make_layer(128, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        layers = [ResidualBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = MiniResNet()
x = torch.randn(2, 3, 32, 32)
with torch.no_grad():
    out = model(x)
print(f"MiniResNet: {sum(p.numel() for p in model.parameters()):,} params")
print(f"Input: {x.shape} -> Output: {out.shape}")

## 5.4 学习率调度策略

不同 scheduler 在不同场景下有各自的优势。

In [ ]:
def get_lr_schedule(scheduler_cls, lr=0.1, epochs=30, **kwargs):
    model = nn.Linear(1, 1)
    opt = optim.SGD(model.parameters(), lr=lr)
    scheduler = scheduler_cls(opt, **kwargs)
    lrs = []
    for _ in range(epochs):
        lrs.append(opt.param_groups[0]["lr"])
        scheduler.step()
    return lrs

schedulers = {
    "StepLR (step=10, gamma=0.5)": get_lr_schedule(optim.lr_scheduler.StepLR, step_size=10, gamma=0.5),
    "MultiStepLR (milestones=[10,20])": get_lr_schedule(optim.lr_scheduler.MultiStepLR, milestones=[10, 20], gamma=0.1),
    "CosineAnnealingLR": get_lr_schedule(optim.lr_scheduler.CosineAnnealingLR, T_max=30),
    "CosineAnnealingWarmRestarts": get_lr_schedule(optim.lr_scheduler.CosineAnnealingWarmRestarts, T_0=10, T_mult=2),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, lrs) in zip(axes.flat, schedulers.items()):
    ax.plot(lrs, "o-", markersize=3)
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Learning Rate")
    ax.grid(True, alpha=0.3)
plt.suptitle("Learning Rate Schedulers Comparison", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("学习单周期 (OneCycleLR) 示例:")
model = nn.Linear(1, 1)
opt = optim.SGD(model.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, total_steps=30)
lrs = []
for _ in range(30):
    lrs.append(opt.param_groups[0]["lr"])
    scheduler.step()

plt.plot(lrs, "o-", markersize=3)
plt.xlabel("Step")
plt.ylabel("Learning Rate")
plt.title("OneCycleLR (warm-up + decay)")
plt.grid(True, alpha=0.3)
plt.show()

## 5.5 梯度裁剪 (Gradient Clipping)

限制梯度的范数，防止**梯度爆炸**，对 RNN 和深层网络尤其重要。

In [ ]:
model = MiniResNet().to(device)
x = torch.randn(2, 3, 32, 32).to(device)
y = torch.randint(0, 10, (2,)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

output = model(x)
loss = criterion(output, y)
optimizer.zero_grad()
loss.backward()

total_norm_before = 0
for p in model.parameters():
    if p.grad is not None:
        total_norm_before += p.grad.data.norm(2).item() ** 2
total_norm_before = total_norm_before ** 0.5

max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

total_norm_after = 0
for p in model.parameters():
    if p.grad is not None:
        total_norm_after += p.grad.data.norm(2).item() ** 2
total_norm_after = total_norm_after ** 0.5

print(f"Gradient norm before clipping: {total_norm_before:.4f}")
print(f"Gradient norm after clipping:  {total_norm_after:.4f} (max={max_norm})")

optimizer.step()

## 5.6 实战：CIFAR-10 上对比技术效果

对比四种配置：
1. 基础 CNN (无 BN, 无残差)
2. + BatchNorm
3. + BatchNorm + 残差连接
4. + BatchNorm + 残差 + 数据增强 + 学习率调度

In [ ]:
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

train_transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

full_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform_aug)
full_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_size = 45000
val_size = 5000
train_set, val_set = torch.utils.data.random_split(full_train, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(full_test, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
def quick_train(model, train_ldr, val_ldr, epochs=15, lr=0.01):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        for data, target in train_ldr:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), target)
            loss.backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in val_ldr:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += data.size(0)
    return correct / total

class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

print("Training models for comparison...")
print("-" * 50)

acc_baseline = quick_train(BaselineCNN(), train_loader, val_loader)
print(f"Baseline CNN:                     Val Acc = {acc_baseline:.2%}")

acc_resnet = quick_train(MiniResNet(), train_loader, val_loader)
print(f"MiniResNet (BN + Residual):        Val Acc = {acc_resnet:.2%}")

print("-" * 50)
print(f"Improvement: +{(acc_resnet - acc_baseline)*100:.1f}% absolute")

## 5.7 单元小结

| 技术 | 作用 | 何时使用 |
|------|------|---------|
| **BatchNorm** | 稳定训练，加速收敛 | 几乎所有 CNN |
| **Dropout** | 防止过拟合 | 全连接层常用，CNN 中用得少 |
| **残差连接** | 训练更深网络 | 深度 > 20 层时必用 |
| **CosineAnnealingLR** | 平滑降低学习率 | 推荐的默认调度器 |
| **梯度裁剪** | 防止梯度爆炸 | RNN/Transformer/大 batch |

### 思考题
1. 为什么 BatchNorm 对 batch_size 敏感？小 batch 时有什么替代方案？
2. ResNet 的 shortcut 连接在反向传播时有什么作用？
3. Dropout 的 p=0.5 意味着什么？（注意 PyTorch 的 p 含义）